In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests

In [1]:
#Breaker Signal Code - Dynamic
#Swings detector 
Ticker_df = pd.read_csv("NSE_Symbols.csv")

def telegram_alert(message):
        url = f'https://api.telegram.org/bot8520982329:AAGr_PfQPzUFI2zQDWJ41TAtmCobla2JcZY/sendMessage'
        Data = {'chat_id': '-5245957606',
        'text': message
        }
        requests.post(url,data=Data) 

Alert_List = []

for Ticker in Ticker_df['Symbol']:
    position = -6
    period = round(((1/7)*-position)+10)
    BTC = yf.Ticker(Ticker).history(period=f'{period}d', interval='1h' )
    if BTC.empty:
            print(f'No data found for {Ticker}')
            continue

    BTC['Swing_High'] = np.where((BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),"Swing High","NA")
    BTC['Swing_Low'] = np.where((BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1)),"Swing Low","NA")

    BTC['Swing_High_Value'] = np.where(BTC['Swing_High']=="Swing High",BTC['High'],np.nan)
    BTC['Swing_Low_Value'] = np.where(BTC['Swing_Low']=="Swing Low",BTC['Low'],np.nan)

    BTC['Last_Swing_High'] = BTC['Swing_High_Value'].ffill()
    BTC['Last_Swing_Low'] = BTC['Swing_Low_Value'].ffill()

    BTC['Prev_Swing_High'] = BTC['Last_Swing_High'].shift(1)
    BTC['Prev_Swing_Low'] = BTC['Last_Swing_Low'].shift(1)

    conditions = [
        (BTC['Swing_High']=='Swing High')&
            (BTC['Last_Swing_High']>BTC['Prev_Swing_High']),

            (BTC['Swing_High']=='Swing High')&
            (BTC['Last_Swing_High']<BTC['Prev_Swing_High']),

            (BTC['Swing_Low']=='Swing Low')&
            (BTC['Last_Swing_Low']>BTC['Prev_Swing_Low']),

            (BTC['Swing_Low']=='Swing Low')&
            (BTC['Last_Swing_Low']<BTC['Prev_Swing_Low']),
        ]

    labels = ['Higher High','Lower High','Higher Low','Lower Low']

    BTC['Master_Swing'] = np.select(conditions,labels,default="")

        #Swings pattern detector 


        #Combined Signal (Swing High/Low)
    conditions = [
            (BTC['High']>BTC['High'].shift(-1))&(BTC['High']>BTC['High'].shift(1)),

            (BTC['Low']<BTC['Low'].shift(-1))&(BTC['Low']<BTC['Low'].shift(1))
        ] 

    signals = ['Swing High','Swing Low']

    BTC['Main_Signal'] = np.select(conditions,signals,default="")

    Subset_df = BTC[BTC['Main_Signal'] !=''].copy()

    Subset_df['Breaker_Signal'] = (
            (Subset_df['Main_Signal'] == 'Swing High')&
            (Subset_df['Main_Signal'].shift(1)=='Swing Low')&
            (Subset_df['Main_Signal'].shift(2)=='Swing High')&
            (Subset_df['Main_Signal'].shift(3)=='Swing Low')&
            (Subset_df['High']>Subset_df['High'].shift(2))&
            (Subset_df['Low'].shift(1)<Subset_df['Low'].shift(3))
            )
    BTC["Breaker_Signal"] = False

    BTC.loc[Subset_df.index,'Breaker_Signal'] = Subset_df['Breaker_Signal']
        #In BTC using the index of subset df, fill the Breaker signal column in BTC with values of subset_df["breaker signal"]

    
    Breaker_Signal = BTC['Breaker_Signal'].iloc[position]
    
    if Breaker_Signal == True:
        # Append both ticker and timestamp (index) as a string
        alert_time = BTC.index[position]
        Alert_List.append(f"{Ticker} at {alert_time}")  
    else:
            print(f'No alert for {Ticker}')

if Alert_List:
        message = (
                "Breaker Signal in the following:\n\n"
                + f"Candle: {position}\n\n"
                +"\n".join(Alert_List) #Convert the list into string and print them each on new line \n 
        )
        telegram_alert(message)



NameError: name 'pd' is not defined

In [36]:
#Breaker Signal Code - Dynamic (Improved)
# 1h bulk download + optional 15min FVG overlap check within the breaker window

Ticker_df    = pd.read_csv('NSE_Symbols.csv')
tickers_list = Ticker_df['Symbol'].tolist()

def telegram_alert(message):
    url = 'https://api.telegram.org/bot8520982329:AAGr_PfQPzUFI2zQDWJ41TAtmCobla2JcZY/sendMessage'
    requests.post(url, data={'chat_id': '-5245957606', 'text': message})

position = -7
period   = round(((1/7) * -position) + 10)

# ── Bulk 1h download ──────────────────────────────────────────────────────────
raw_data    = yf.download(tickers_list, period=f'{period}d', interval='1h',
                          group_by='ticker', auto_adjust=True, progress=False)
multi_index = isinstance(raw_data.columns, pd.MultiIndex)

Alert_List = []   # list of (Ticker, has_15m_fvg)
alert_time = None

for Ticker in tickers_list:
    try:
        BTC = (raw_data[Ticker] if multi_index else raw_data).copy()
    except KeyError:
        continue

    BTC.dropna(how='all', inplace=True)
    if BTC.empty or len(BTC) < abs(position):
        continue

    # ── Swing + Breaker detection on 1h ──────────────────────────────────────
    BTC['Main_Signal'] = np.select(
        [(BTC['High'] > BTC['High'].shift(-1)) & (BTC['High'] > BTC['High'].shift(1)),
         (BTC['Low']  < BTC['Low'].shift(-1))  & (BTC['Low']  < BTC['Low'].shift(1))],
        ['Swing High', 'Swing Low'], default='')

    Subset_df = BTC[BTC['Main_Signal'] != ''].copy()
    if len(Subset_df) < 4:
        continue

    Subset_df['Breaker_Signal'] = (
        (Subset_df['Main_Signal'] == 'Swing High') &
        (Subset_df['Main_Signal'].shift(1) == 'Swing Low') &
        (Subset_df['Main_Signal'].shift(2) == 'Swing High') &
        (Subset_df['Main_Signal'].shift(3) == 'Swing Low') &
        (Subset_df['High'] > Subset_df['High'].shift(2)) &
        (Subset_df['Low'].shift(1) < Subset_df['Low'].shift(3))
    )
    BTC['Breaker_Signal'] = False
    BTC.loc[Subset_df.index, 'Breaker_Signal'] = Subset_df['Breaker_Signal']

    if not BTC['Breaker_Signal'].iloc[position]:
        continue

    target_idx = BTC.index[position]
    if alert_time is None:
        alert_time = target_idx

    # ── Locate T2 (first swing high in the breaker chain) ────────────────────
    t2_ts = None
    for idx in Subset_df.index[Subset_df['Breaker_Signal']]:
        if idx == target_idx:
            p = Subset_df.index.get_loc(idx)
            if p >= 3:
                t2_ts = Subset_df.index[p - 2]
            break

    # ── Check 15min data for FVG overlap within [T2, T0] ─────────────────────
    has_15m_fvg = False
    if t2_ts is not None:
        try:
            M15 = yf.Ticker(Ticker).history(period=f'{period}d', interval='15m')
            if not M15.empty:
                # Slice to structural window matching the 1h breaker chain
                M15_win = M15[(M15.index >= t2_ts) & (M15.index <= target_idx)].copy()
                if len(M15_win) >= 3:
                    M15_win['FVG_Signal'] = np.select(
                        [(M15_win['Low']  > M15_win['High'].shift(2)),
                         (M15_win['High'] < M15_win['Low'].shift(2))],
                        ['Bullish_FVG', 'Bearish_FVG'], default='NA')

                    M15_win['T0_Bull_Low']     = np.where(M15_win['FVG_Signal'] == 'Bullish_FVG', M15_win['Low'],            np.nan)
                    M15_win['T1_Bull_High']    = np.where(M15_win['FVG_Signal'] == 'Bullish_FVG', M15_win['High'].shift(2),  np.nan)
                    M15_win['T0_Bearish_High'] = np.where(M15_win['FVG_Signal'] == 'Bearish_FVG', M15_win['High'],           np.nan)
                    M15_win['T1_Bearish_Low']  = np.where(M15_win['FVG_Signal'] == 'Bearish_FVG', M15_win['Low'].shift(2),   np.nan)

                    Sub_15 = M15_win[M15_win['FVG_Signal'] != 'NA'].copy()
                    Sub_15['FVG_Overlap'] = np.select(
                        [(Sub_15['FVG_Signal'] == 'Bullish_FVG') &
                         (Sub_15['FVG_Signal'].shift(1) == 'Bearish_FVG') &
                         (Sub_15['T0_Bull_Low'] > Sub_15['T0_Bearish_High'].shift(1)) &
                         (Sub_15['T1_Bearish_Low'].shift(1) > Sub_15['T1_Bull_High'])],
                        ['FVG_Overlap'], default='NA')

                    has_15m_fvg = (Sub_15['FVG_Overlap'] == 'FVG_Overlap').any()
        except Exception:
            pass

    Alert_List.append((Ticker, has_15m_fvg))

# ── Send alert ────────────────────────────────────────────────────────────────
if Alert_List:
    # Tickers with 15min FVG confluence listed first
    Alert_List.sort(key=lambda x: x[1], reverse=True)
    lines = [
        f"{ticker}  [+15m FVG Overlap]" if has_fvg else ticker
        for ticker, has_fvg in Alert_List
    ]
    alert_time_ist = alert_time.tz_convert('Asia/Kolkata') if hasattr(alert_time, 'tz_convert') else alert_time
    message = (
        "Breaker Signals\n\n"
        f"Candle: {position}\n"
        f"Time: {alert_time_ist}\n\n"
        + "\n".join(lines)
    )
    telegram_alert(message)
    print(message)
else:
    no_signal_msg = f"Breaker Scanner ran — no signals at candle {position}."
    telegram_alert(no_signal_msg)
    print(no_signal_msg)


Breaker Signals

Candle: -7
Time: 2026-04-24 09:15:00+05:30

COLPAL.NS  [+15m FVG Overlap]
RBLBANK.NS  [+15m FVG Overlap]
BRITANNIA.NS
GODREJPROP.NS
HDFCAMC.NS
HEROMOTOCO.NS
IRCTC.NS
M&M.NS
MARUTI.NS
SBIN.NS
